In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------
# 1. 定义超简单的注意力层
# ----------------------
class SimpleAttention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # 这就是你问的 W_Q, W_K, W_V
        self.Wq = nn.Linear(input_dim, input_dim)
        self.Wk = nn.Linear(input_dim, input_dim)
        self.Wv = nn.Linear(input_dim, input_dim)

    def forward(self, X):
        # 1. 生成 Q, K, V
        Q = self.Wq(X)
        K = self.Wk(X)
        V = self.Wv(X)

        # 2. 计算注意力分数
        attn_score = torch.matmul(Q, K.transpose(-1, -2))
        
        # 3. 归一化得到权重
        attn_weight = torch.softmax(attn_score, dim=-1)
        
        # 4. 加权得到注意力输出
        out = torch.matmul(attn_weight, V)
        return out, attn_weight

# ----------------------
# 2. 定义完整分类模型
# ----------------------
class AnimalClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=2):
        super().__init__()
        self.attention = SimpleAttention(input_dim)
        self.fc = nn.Linear(input_dim, 2)  # 最后输出2类：狼/兔子

    def forward(self, X):
        attn_out, weight = self.attention(X)
        # 把注意力输出压扁，用于分类
        out = attn_out.mean(dim=1)
        out = self.fc(out)
        return out, weight

# ----------------------
# 3. 构造数据集（你要的 X → Y）
# ----------------------
# X = [体型, 颜色]
# 体型：0=小，1=大
# 颜色：0=灰，1=白
# 狼：[1,0] = 大+灰 → 标签0
# 兔子：[0,1] = 小+白 → 标签1
X = torch.tensor([
    [[1, 0]],  # 大+灰 → 狼
    [[0, 1]],  # 小+白 → 兔子
    [[1, 0]],
    [[0, 1]],
], dtype=torch.float32)

Y = torch.tensor([0, 1, 0, 1])  # 真实答案

# ----------------------
# 4. 训练模型
# ----------------------
model = AnimalClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 开始训练
for epoch in range(100):
    optimizer.zero_grad()
    output, weight = model(X)
    loss = criterion(output, Y)
    loss.backward()  # 反向传播 → 自动更新 Wq,Wk,Wv
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.3f}")

# ----------------------
# 5. 测试
# ----------------------
print("\n==== 测试结果 ====")
test_X = torch.tensor([[[1,0]], [[0,1]]], dtype=torch.float32)
with torch.no_grad():
    pred, weight = model(test_X)
    print("输入[大,灰] → 预测类别：", pred.argmax(1)[0].item(), "（0=狼）")
    print("输入[小,白] → 预测类别：", pred.argmax(1)[1].item(), "（1=兔子）")

print("\n注意力权重（模型学会关注什么特征）：")

Epoch 0 | Loss: 0.732
Epoch 10 | Loss: 0.651
Epoch 20 | Loss: 0.571
Epoch 30 | Loss: 0.479
Epoch 40 | Loss: 0.372
Epoch 50 | Loss: 0.263
Epoch 60 | Loss: 0.171
Epoch 70 | Loss: 0.107
Epoch 80 | Loss: 0.068
Epoch 90 | Loss: 0.046

==== 测试结果 ====
输入[大,灰] → 预测类别： 0 （0=狼）
输入[小,白] → 预测类别： 1 （1=兔子）

注意力权重（模型学会关注什么特征）：
tensor([[[1.]],

        [[1.]]])
